# CycleGAN — Flip Rate on Training Split (H→P)

Evaluates CycleGAN-generated healthy→pneumonia images using CheXNet as a fixed arbiter,
restricted to the **training split** healthy images.

**No Kaggle download required** — originals are loaded from `data/healthy_images/`,
synthetics from `data/generated/healthy_to_pneumonia/`.

Output: `results/cyclegan_h2p_train_scores.csv` — used by notebook 1.5 to select
only flipped synthetic images as augmentation.

## 0. Path configuration

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path("../").resolve()
sys.path.insert(0, str(PROJECT_ROOT))
sys.path.insert(0, str(PROJECT_ROOT / "utils"))
sys.path.insert(0, str(PROJECT_ROOT / "models" / "Classifier"))

HEALTHY_DIR  = PROJECT_ROOT / "data" / "healthy_images"
H2P_DIR      = PROJECT_ROOT / "data" / "generated" / "healthy_to_pneumonia"
TRAIN_CSV    = PROJECT_ROOT / "data" / "train_split.csv"
CHEXNET_CKPT = PROJECT_ROOT / "models" / "Classifier" / "chexnet.pth.tar"
RESULTS_DIR  = PROJECT_ROOT / "results"
RESULTS_DIR.mkdir(exist_ok=True)

print("HEALTHY_DIR  :", HEALTHY_DIR,  "| exists:", HEALTHY_DIR.exists())
print("H2P_DIR      :", H2P_DIR,      "| images:", len(list(H2P_DIR.glob("*.png"))))
print("TRAIN_CSV    :", TRAIN_CSV,    "| exists:", TRAIN_CSV.exists())
print("CHEXNET_CKPT :", CHEXNET_CKPT, "| exists:", CHEXNET_CKPT.exists())

HEALTHY_DIR  : C:\Users\User\Documents\0Unicamp\IA376N\Projeto\dgm-2026.1\projects\ecgpcx-ray\data\healthy_images | exists: True
H2P_DIR      : C:\Users\User\Documents\0Unicamp\IA376N\Projeto\dgm-2026.1\projects\ecgpcx-ray\data\generated\healthy_to_pneumonia | images: 60353
TRAIN_CSV    : C:\Users\User\Documents\0Unicamp\IA376N\Projeto\dgm-2026.1\projects\ecgpcx-ray\data\train_split.csv | exists: True
CHEXNET_CKPT : C:\Users\User\Documents\0Unicamp\IA376N\Projeto\dgm-2026.1\projects\ecgpcx-ray\models\Classifier\chexnet.pth.tar | exists: True


## 1. Load CheXNet (fixed arbiter)

In [2]:
import torch
from classifier import CheXNet
from device import DEVICE

chexnet = CheXNet(CHEXNET_CKPT).to(DEVICE)
print(f"CheXNet loaded on {DEVICE} | eval={not chexnet.training} | trainable params=0")

CheXNet loaded on cuda | eval=True | trainable params=0


## 2. Load training split and match with generated images

Only healthy images (`Label == 0`) from the training split are evaluated.
Generated images are matched by filename — images not in the training split are excluded
to prevent leakage.

In [3]:
import pandas as pd

train_df      = pd.read_csv(TRAIN_CSV)
healthy_train = train_df[train_df["Label"] == 0].reset_index(drop=True)

print(f"Training split rows:   {len(train_df)}")
print(f"Healthy (Label == 0):  {len(healthy_train)}")
print(f"Generated H2P images:  {len(list(H2P_DIR.glob('*.png')))}")

Training split rows:   43450
Healthy (Label == 0):  42436
Generated H2P images:  60353


In [4]:
h2p_files           = {p.name: p for p in H2P_DIR.glob("*.png")}
healthy_train_names = set(healthy_train["Image Index"])

# Keep only generated images whose source is a training-split healthy image
h2p_pairs = [
    (name, h2p_files[name])
    for name in healthy_train_names
    if name in h2p_files
]

print(f"Usable H→P pairs: {len(h2p_pairs)}")

Usable H→P pairs: 42436


## 3. Helper functions

In [5]:
from PIL import Image
from torchvision import transforms

resize_224 = transforms.Resize((224, 224), antialias=True)
to_tensor  = transforms.ToTensor()


def original_tensor(nih_filename: str) -> torch.Tensor:
    """Load a healthy original from data/healthy_images/ as (1, 1, 224, 224)."""
    path = HEALTHY_DIR / nih_filename
    if not path.exists():
        raise FileNotFoundError(f"{nih_filename} not found in {HEALTHY_DIR}")
    return resize_224(to_tensor(Image.open(path).convert("L"))).unsqueeze(0)


def synthetic_tensor(path: Path) -> torch.Tensor:
    """Load a CycleGAN-generated image as (1, 1, 224, 224)."""
    return resize_224(to_tensor(Image.open(path).convert("L"))).unsqueeze(0)


@torch.inference_mode()
def score(tensor: torch.Tensor) -> float:
    """P(pneumonia) for a single (1, 1, 224, 224) tensor."""
    return chexnet.predict_pneumonia(tensor.to(DEVICE)).item()


# Threshold from notebook 2.0 (derived in notebook 1.6 using val set)
THRESHOLD = 0.3414328098297119
print(f"CheXNet threshold: {THRESHOLD}")

CheXNet threshold: 0.3414328098297119


## 4. Evaluate Healthy → Pneumonia (training split)

- **Original**: healthy image from training → P(pneumonia) expected low
- **Translated**: CycleGAN(healthy) → P(pneumonia) expected high
- **Flip**: original < threshold **and** translated ≥ threshold

In [6]:
from tqdm.notebook import tqdm

h2p_results = []

for name, synth_path in tqdm(h2p_pairs, desc="H→P (train)"):
    p_orig  = score(original_tensor(name))
    p_synth = score(synthetic_tensor(synth_path))

    h2p_results.append({
        "image"       : name,
        "p_original"  : p_orig,
        "p_translated": p_synth,
        "delta"       : p_synth - p_orig,
        "flip"        : int(p_orig < THRESHOLD and p_synth >= THRESHOLD),
    })

h2p_df = pd.DataFrame(h2p_results)
print(f"Evaluated: {len(h2p_df)} pairs")

H→P (train):   0%|          | 0/42436 [00:00<?, ?it/s]

Evaluated: 42436 pairs


## 5. Results summary

In [7]:
n_flipped   = int(h2p_df["flip"].sum())
n_total     = len(h2p_df)
flip_rate   = h2p_df["flip"].mean()

print(f"{'='*45}")
print(f"  Healthy → Pneumonia  (training split)")
print(f"{'='*45}")
print(f"  Pairs evaluated    : {n_total}")
print(f"  Flip Rate          : {flip_rate:.3f}  ({n_flipped}/{n_total})")
print(f"  Δ Confidence       : {h2p_df['delta'].mean():+.3f} ± {h2p_df['delta'].std():.3f}")
print(f"  Mean P (original)  : {h2p_df['p_original'].mean():.3f}")
print(f"  Mean P (translated): {h2p_df['p_translated'].mean():.3f}")
print(f"\n  → {n_flipped} synthetic images will be used for augmentation in notebook 1.5")

  Healthy → Pneumonia  (training split)
  Pairs evaluated    : 42436
  Flip Rate          : 0.147  (6230/42436)
  Δ Confidence       : +0.046 ± 0.099
  Mean P (original)  : 0.313
  Mean P (translated): 0.359

  → 6230 synthetic images will be used for augmentation in notebook 1.5


## 6. Visualization

In [8]:
import matplotlib.pyplot as plt
import seaborn as sns

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
fig.suptitle("CycleGAN H→P — CheXNet evaluation on training split", fontsize=13)

ax = axes[0]
sns.kdeplot(h2p_df["p_original"],   ax=ax, fill=True, color="steelblue",
            alpha=0.5, label="original (healthy)", common_norm=False)
sns.kdeplot(h2p_df["p_translated"], ax=ax, fill=True, color="tomato",
            alpha=0.5, label="translated (synthetic)", common_norm=False)
ax.axvline(THRESHOLD, color="black", linestyle="--", linewidth=1, label=f"threshold={THRESHOLD:.3f}")
ax.set_xlabel("P(pneumonia)")
ax.set_ylabel("Density")
ax.set_title("P(pneumonia) before vs after translation")
ax.legend()

ax = axes[1]
sns.histplot(h2p_df["delta"], ax=ax, bins=40, color="mediumpurple", alpha=0.7)
ax.axvline(0, color="black", linestyle="--", linewidth=1)
ax.axvline(h2p_df["delta"].mean(), color="tomato", linestyle="-", linewidth=1.5,
           label=f"mean = {h2p_df['delta'].mean():+.3f}")
ax.set_xlabel("Δ P(pneumonia) = P(translated) − P(original)")
ax.set_ylabel("Count")
ax.set_title(f"Confidence Change  |  Flip Rate = {flip_rate:.3f}")
ax.legend()

plt.tight_layout()
plt.savefig(RESULTS_DIR / "cyclegan_flip_rate_train.png", dpi=120)
plt.show()

ModuleNotFoundError: No module named 'seaborn'

## 7. Save results

The CSV is consumed by **notebook 1.5**: it filters `gen_img_paths` to only the rows
where `flip == 1`, using only the synthetic images that CheXNet agrees look like pneumonia.

In [9]:
import json

out_csv = RESULTS_DIR / "cyclegan_h2p_train_scores.csv"
h2p_df.to_csv(out_csv, index=False)
print(f"Scores saved to: {out_csv}")

summary = {
    "split"        : "train",
    "threshold"    : THRESHOLD,
    "n_pairs"      : n_total,
    "n_flipped"    : n_flipped,
    "flip_rate"    : round(flip_rate, 4),
    "delta_mean"   : round(h2p_df["delta"].mean(), 4),
    "delta_std"    : round(h2p_df["delta"].std(),  4),
}

with open(RESULTS_DIR / "cyclegan_flip_rate_train.json", "w") as f:
    json.dump(summary, f, indent=2)

print(json.dumps(summary, indent=2))

Scores saved to: C:\Users\User\Documents\0Unicamp\IA376N\Projeto\dgm-2026.1\projects\ecgpcx-ray\results\cyclegan_h2p_train_scores.csv
{
  "split": "train",
  "threshold": 0.3414328098297119,
  "n_pairs": 42436,
  "n_flipped": 6230,
  "flip_rate": 0.1468,
  "delta_mean": 0.0465,
  "delta_std": 0.0991
}
